# R2-05 — Per-path evaluation on ALL 5 MSILN site1/B1 test paths (camera-ready, ICINCO 2026 paper #122)

Produces the data for the new Figure 6c: test MAE per path for **all 5 MSILN test paths**, for

1. **Ours** — the R1-01 fusion checkpoints from Drive (`runs/msiln_site1_b1_seed*/model_last.pt`),
   no retraining; all 3 seeds evaluated (figure uses seed 42).
2. **PDR-from-start** — deterministic pedestrian dead reckoning (competition step detection),
   anchored at the first ground-truth waypoint; the repo routine returns per-path MAE directly.

Result cells write JSON to Drive under `r2_05/` and skip themselves on re-run. GPU runtime
recommended (T4); total a few minutes.


In [ ]:
# ==== Parameters ====
SEEDS = [42, 7, 123]        # fusion checkpoints to evaluate (from R1-01)
FIGURE_SEED = 42            # figures use this seed per the Sec 5.4 convention
K = 4                       # n_instants, paper config (must match R1-01)
BATCH = 128                 # paper config
MBL = False                 # modality_balanced_loss off = paper config
DRIVE_DIR = "navlori_camera_ready"


In [ ]:
# ==== Google Drive ====
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
OUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_DIR
R205 = OUT_ROOT / "r2_05"
R205.mkdir(parents=True, exist_ok=True)
print("results root:", R205)
ckpts = sorted((OUT_ROOT / "runs").glob("msiln_site1_b1_seed*/model_last.pt"))
print("fusion checkpoints found:", [str(p.parent.name) for p in ckpts])
assert len(ckpts) >= 3, "R1-01 MSILN checkpoints missing from Drive - tell Claude"


In [ ]:
# ==== Clone repo + submodules + self-healing imports ====
import os, sys, subprocess, json, time, random, io, re
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (CPU is fine for this notebook, just slower)")

REPO = Path("/content/navlori-fusion")
if not REPO.exists():
    subprocess.check_call(["git", "clone", "--depth", "1",
                           "https://github.com/moebachar/navlori-fusion.git", str(REPO)])
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

def _run(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip(): print(out.strip()[-2000:])
    return r.returncode

# ronin + dpvo: eager imports in src.pipeline.encoders/baselines;
# indoor_location_competition_20: step detection used by PDR-from-start.
SUBMODULES = {
    "external_methods/ronin": ("https://github.com/Sachini/ronin",
                               "805b7f0f28bb164ce89ada9ac05a9470dbe3d715",
                               "source/model_resnet1d.py"),
    "external_methods/dpvo": ("https://github.com/princeton-vl/DPVO",
                              "859bbbfdac6c6185f345003b3c473901fcd13ace",
                              "dpvo/extractor.py"),
    "external_methods/indoor_location_competition_20": (
                              "https://github.com/location-competition/indoor-location-competition-20",
                              "8ab177cd7be2700442714b7d13fa452bedc685f5",
                              "compute_f.py"),
}
print("-- git submodule update --init --")
_run(["git", "submodule", "update", "--init"] + list(SUBMODULES), cwd=str(REPO))
import shutil
for path, (url, sha, marker) in SUBMODULES.items():
    d = REPO / path
    if not (d / marker).exists():
        print(f"-- submodule route did not materialize {path}; direct clone fallback --")
        if d.exists():
            shutil.rmtree(d, ignore_errors=True)
        _run(["git", "clone", url, str(d)])
        _run(["git", "checkout", sha], cwd=str(d))
    assert (d / marker).exists(), f"{path}/{marker} STILL missing - save a copy to GitHub and tell Claude"
    print(f"OK: {path} ({marker} present)")

PIPNAME = {"omegaconf": "omegaconf", "yaml": "pyyaml", "sklearn": "scikit-learn",
           "cv2": "opencv-python-headless", "mlflow": "mlflow", "torchdiffeq": "torchdiffeq",
           "seaborn": "seaborn", "influxdb_client": "influxdb-client", "plotly": "plotly",
           "quaternion": "numpy-quaternion", "PIL": "pillow", "skimage": "scikit-image"}
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "quaternion"],
               capture_output=True)
_tried = set()
for attempt in range(10):
    try:
        from src.pipeline.fusion.builder import (build_datamodule, build_encoders,
                                                 build_model, build_trainer, load_config)
        from src.pipeline.baselines.msiln_baselines import run_pdr_from_start_msiln
        print("imports OK"); break
    except ModuleNotFoundError as e:
        pkg = e.name.split(".")[0]
        if pkg in _tried:
            raise RuntimeError(
                f"module '{e.name}' still missing after installing '{PIPNAME.get(pkg, pkg)}' - "
                "save a copy to GitHub and tell Claude.") from e
        _tried.add(pkg)
        pip = PIPNAME.get(pkg, pkg)
        print("missing module:", e.name, "-> pip install", pip)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip])
        if r.returncode != 0:
            raise RuntimeError(
                f"'{e.name}' is not pip-installable - likely a repo-local module. "
                "Save a copy of this notebook to GitHub and tell Claude.") from e
else:
    raise RuntimeError("imports still failing after installs - send Claude the error above")

import numpy as np

def set_global_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Data — MSILN site1/B1

Restored from the Drive cache left by the R1-01 notebook; the starter-repo conversion
below only runs if the cache is gone.


In [ ]:
# ==== MSILN: restore from Drive cache (starter-repo fallback) ====
import shutil
DATA_CACHE = OUT_ROOT / "data_cache"

def run_logged(cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout or "") + (r.stderr or "")
    if out.strip(): print(out.strip()[-3000:])
    return r.returncode

def drive_restore(name, marker="metadata.json"):
    dst = REPO / "data" / name
    if (dst / marker).exists(): return "already in session"
    src = DATA_CACHE / name
    if (src / marker).exists():
        shutil.copytree(src, dst, dirs_exist_ok=True); return "restored from Drive cache"
    return None

st = drive_restore("msiln_site1_b1")
if st:
    print("MSILN:", st)
else:
    STARTER = Path("/content/indoor-location-competition-20")
    if not STARTER.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/location-competition/indoor-location-competition-20.git",
            str(STARTER)])
    if run_logged([sys.executable, "scripts/convert_msiln.py", "--msiln-root", str(STARTER),
                   "--site", "site1", "--floor", "B1", "--out-root", "data"]) != 0:
        raise RuntimeError("convert_msiln.py failed - output above")
n_paths = len(list((REPO / "data" / "msiln_site1_b1").glob("path_*")))
print(f"MSILN ready: {n_paths} paths (expected 133)")


## Method 1 — Ours: per-path MAE from the R1-01 checkpoints (no retraining)

Rebuilds the exact R1-01 trainer, loads each seed's `model_last.pt`, predicts the test
split and groups errors by path. Overall val/test are cross-checked against the R1-01
`summary.json` (they must match to a few millimeters).


In [ ]:
# ==== Ours per-path (3 seeds) ====
for seed in SEEDS:
    out_p = R205 / f"ours_seed{seed}.json"
    if out_p.exists():
        d = json.loads(out_p.read_text())
        print(f"skip ours seed {seed}: test {d['overall']['test_mae_m']:.2f} m "
              f"({len(d['per_path_test'])} paths)")
        continue
    print(f"\n===== ours seed {seed} =====", flush=True)
    set_global_seed(seed)
    cfg = load_config("msiln_site1_b1")
    cfg.temporal.n_instants = K
    cfg.train.modality_balanced_loss = MBL
    cfg.data.batch_size = BATCH
    dm = build_datamodule(cfg)
    encs, vision = build_encoders(cfg, dm)
    model = build_model(cfg, encs)
    trainer = build_trainer(cfg, model, dm, run_dir=f"/content/tmp_eval_msiln_seed{seed}")
    ckpt = OUT_ROOT / "runs" / f"msiln_site1_b1_seed{seed}" / "model_last.pt"
    sd = torch.load(ckpt, map_location=trainer.device)
    trainer.model.load_state_dict(sd)
    print("checkpoint loaded:", ckpt.name)

    res = {"seed": seed, "overall": {}, "per_path_test": {}}
    preds_v, tgts_v = trainer.predict("val")
    res["overall"]["val_mae_m"] = float((preds_v - tgts_v).norm(dim=1).mean())
    preds_t, tgts_t = trainer.predict("test")
    res["overall"]["test_mae_m"] = float((preds_t - tgts_t).norm(dim=1).mean())

    summ = json.loads((OUT_ROOT / "runs" / f"msiln_site1_b1_seed{seed}" / "summary.json").read_text())
    dv = abs(res["overall"]["val_mae_m"] - summ["val_mae_m"])
    dt = abs(res["overall"]["test_mae_m"] - summ["test_mae_m"])
    print(f"overall: val {res['overall']['val_mae_m']:.3f} m (R1-01: {summ['val_mae_m']:.3f}), "
          f"test {res['overall']['test_mae_m']:.3f} m (R1-01: {summ['test_mae_m']:.3f})")
    if max(dv, dt) > 0.05:
        print("!! WARNING: overall MAE deviates from the R1-01 summary by "
              f"{max(dv, dt):.3f} m - tell Claude before using these numbers")

    err = (preds_t - tgts_t).norm(dim=1).numpy()
    ds_test = trainer.dm.test_ds
    pids = np.array([r["path_id"] for r in ds_test._gt_rows])[:len(err)]
    for pid in sorted(np.unique(pids)):
        m = pids == pid
        res["per_path_test"][str(int(pid))] = {
            "mae": float(err[m].mean()), "n": int(m.sum())}
        print(f"  path_{int(pid):02d}: MAE {err[m].mean():.3f} m  n={int(m.sum())}")
    out_p.write_text(json.dumps(res, indent=2))
    print("wrote", out_p.name)
    del trainer, model, dm, encs
    torch.cuda.empty_cache()
print("\nours: done")


## Method 2 — PDR-from-start: per-path MAE

The repo routine (`run_pdr_from_start_msiln`) is deterministic and already returns one
MAE per path; its split means are cross-checked against Table 4 (val 16.88, test 12.49).


In [ ]:
# ==== PDR-from-start per-path ====
out_p = R205 / "pdr_perpath.json"
if out_p.exists():
    d = json.loads(out_p.read_text())
    print(f"skip pdr: test {d['overall']['test_mae_m']:.2f} m ({len(d['per_path_test'])} paths)")
else:
    summary = run_pdr_from_start_msiln(verbose=True)
    res = {"overall": {"val_mae_m": summary["val_mae"], "test_mae_m": summary["test_mae"],
                       "val_n_paths": summary["val_n_paths"], "test_n_paths": summary["test_n_paths"]},
           "per_path_test": {str(r["path_id"]): {"mae": r["mae"], "n_steps": r["n_steps"],
                                                  "final_drift": r["final_drift"]}
                             for r in summary["per_path"]["test"]},
           "per_path_val": {str(r["path_id"]): {"mae": r["mae"], "n_steps": r["n_steps"]}
                            for r in summary["per_path"]["val"]}}
    for k, v in sorted(res["per_path_test"].items(), key=lambda kv: int(kv[0])):
        print(f"  path_{int(k):02d}: MAE {v['mae']:.3f} m  steps={v['n_steps']}  final drift {v['final_drift']:.2f} m")
    print(f"overall (mean of per-path MAE): val {res['overall']['val_mae_m']:.3f} m (Table 4: 16.88), "
          f"test {res['overall']['test_mae_m']:.3f} m (Table 4: 12.49)")
    out_p.write_text(json.dumps(res, indent=2))
    print("wrote", out_p.name)


## Aggregate — the Figure 6c data (PASTE-BACK)

One row per test path: PDR-from-start, ours seed 42, ours mean and std over 3 seeds,
sample counts and share of the test mass.


In [ ]:
# ==== Aggregate + PASTE-BACK JSON ====
import statistics
ours = {s: json.loads((R205 / f"ours_seed{s}.json").read_text()) for s in SEEDS}
pdr = json.loads((R205 / "pdr_perpath.json").read_text())

pids = sorted(int(p) for p in ours[FIGURE_SEED]["per_path_test"])
table = {}
for pid in pids:
    k = str(pid)
    o = [ours[s]["per_path_test"][k]["mae"] for s in SEEDS]
    table[k] = {
        "pdr": round(pdr["per_path_test"][k]["mae"], 3) if k in pdr["per_path_test"] else None,
        "ours_seed42": round(ours[FIGURE_SEED]["per_path_test"][k]["mae"], 3),
        "ours_mean": round(statistics.mean(o), 3),
        "ours_std": round(statistics.stdev(o), 3),
        "n": ours[FIGURE_SEED]["per_path_test"][k]["n"],
    }
n_tot = sum(r["n"] for r in table.values())
agg = {
    "comment": "R2-05 per-path test MAE (m), MSILN site1/B1, all test paths",
    "figure_seed": FIGURE_SEED,
    "per_path": table,
    "macro_avg": {
        "pdr": round(statistics.mean([r["pdr"] for r in table.values() if r["pdr"] is not None]), 3),
        "ours_seed42": round(statistics.mean([r["ours_seed42"] for r in table.values()]), 3),
    },
    "overall": {
        "pdr_mean_of_paths": {"val": round(pdr["overall"]["val_mae_m"], 3),
                              "test": round(pdr["overall"]["test_mae_m"], 3)},
        "ours": {s: {"val": round(ours[s]["overall"]["val_mae_m"], 3),
                     "test": round(ours[s]["overall"]["test_mae_m"], 3)} for s in SEEDS},
    },
    "wins": {"ours_ahead_seed42": sum(1 for r in table.values()
                                      if r["pdr"] is not None and r["ours_seed42"] < r["pdr"])},
    "path_mass": {k: round(r["n"] / n_tot, 3) for k, r in table.items()},
}
hdr = f"{'path':>5} {'n':>5} {'mass':>6} {'pdr':>7} {'ours42':>7} {'ours m+/-s':>12}"
print(hdr); print("-" * len(hdr))
for pid in pids:
    r = table[str(pid)]
    print(f"{pid:>5} {r['n']:>5} {agg['path_mass'][str(pid)]:>6} {str(r['pdr']):>7} {r['ours_seed42']:>7} "
          f"{r['ours_mean']:>6}+/-{r['ours_std']:<4}")
print("-" * len(hdr))
print("macro:", agg["macro_avg"], " wins:", agg["wins"])
(R205 / "r2_05_perpath.json").write_text(json.dumps(agg, indent=2))
print("\nwrote r2_05_perpath.json to Drive\n")
print("=" * 30, "PASTE-BACK JSON", "=" * 30)
print(json.dumps(agg))
